# Grand Teton stable surface-water mask


In [ ]:

import ee
import geemap
import requests
import osmnx as ox
from shapely.geometry import shape
import json

EE_PROJECT = "teton-classifier"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

## Load the official NPS boundary

The ArcGIS query selects the park with NPS unit code `GRTE` and returns its geometry in WGS 84.

In [15]:
NPS_QUERY_URL = (
    "https://services1.arcgis.com/fBc8EJBxQRMcHlei/arcgis/rest/services/"
    "NPS_Land_Resources_Division_Boundary_and_Tract_Data_Service/"
    "FeatureServer/2/query"
)

params = {
    "where": "UNIT_CODE='GRTE'",
    "outFields": "UNIT_CODE,UNIT_NAME,DATE_EDIT,STATE",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson",
}

response = requests.get(NPS_QUERY_URL, params=params, timeout=60)
response.raise_for_status()
boundary_geojson = response.json()

assert len(boundary_geojson["features"]) == 1, "Expected one GRTE boundary feature"
park = ee.Geometry(boundary_geojson["features"][0]["geometry"])
boundary_geojson["features"][0]["properties"]

{'UNIT_CODE': 'GRTE',
 'UNIT_NAME': 'Grand Teton National Park',
 'DATE_EDIT': 1741824000000,
 'STATE': 'WY'}

In [18]:
boundary_geojson

{'type': 'FeatureCollection',
 'crs': {'type': 'name', 'properties': {'name': 'EPSG:4326'}},
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-110.795300423214, 44.0836225993593],
      [-110.795271956501, 44.0799967130179],
      [-110.795251046417, 44.0763739945409],
      [-110.795230163281, 44.0727512201284],
      [-110.795188870422, 44.0654766726016],
      [-110.795147769803, 44.0582382906369],
      [-110.79512711484, 44.050999880453],
      [-110.795106460775, 44.0437614701373],
      [-110.785084478173, 44.0437634621746],
      [-110.775062497367, 44.0437654264461],
      [-110.765040486918, 44.0437673913631],
      [-110.755018505214, 44.0437693808174],
      [-110.744996488476, 44.0437713089284],
      [-110.734974444788, 44.0437732389766],
      [-110.725060714568, 44.0437751238245],
      [-110.724952401101, 44.0437751393217],
      [-110.720057985073, 44.0437761918404],
      [-110.714930329566, 44.043777068724],
      [-110.7149

## Convert GTNP park boundary to shapely boundary, then query OSM api 

In [22]:
# Convert the official NPS boundary to a Shapely geometry.
park_shape = shape(boundary_geojson["features"][0]["geometry"])

# Query OSM water-area tags.
tags = {
    "natural": "water",
    "waterway": "riverbank",  # Legacy river-surface polygons
    "landuse": "reservoir",   # Legacy reservoir polygons
}

osm_water = ox.features_from_polygon(park_shape, tags)
print(f"Downloaded {len(osm_water):,} candidate OSM features")

Downloaded 336 candidate OSM features


## Exclude non-permanent objects

In [24]:
# Keep polygon geometry only. This excludes stream and river centerlines.
# osm_water = osm_water[
#     osm_water.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
# ].copy()

# Exclude features explicitly identified as non-permanent.
for column in ["intermittent", "seasonal"]:
    if column in osm_water.columns:
        values = osm_water[column].fillna("").astype(str).str.lower()
        osm_water = osm_water[~values.isin(["yes", "true", "1"])]

# Repair invalid geometries and discard empty results.
osm_water["geometry"] = osm_water.geometry.make_valid()
osm_water = osm_water[~osm_water.geometry.is_empty]

print(f"Retained {len(osm_water):,} water polygons")


Retained 266 water polygons


# Convert to GEE image mask

In [25]:
# Export geometry only, avoiding OSM attributes containing NaN or unsupported types.
osm_geojson = json.loads(
    osm_water.geometry
    .to_crs("EPSG:4326")
    .to_json()
)

osm_water_fc = ee.FeatureCollection(osm_geojson)

In [26]:
osm_water_mask = (
    ee.Image(0)
    .byte()
    .paint(osm_water_fc, 1)
    .clip(park)
    .rename("osm_surface_water")
)

In [27]:
Map = geemap.Map()
Map.centerObject(park, 10)

Map.addLayer(
    osm_water_mask.selfMask(),
    {"palette": ["0066ff"]},
    "OSM surface-water polygons",
)

Map.addLayer(
    ee.Image().paint(park, 1, 2),
    {"palette": ["ffcc00"]},
    "NPS boundary",
)

Map

Map(center=[43.81796874784758, -110.70554261842562], controls=(WidgetControl(options=['position', 'transparent…

## Export

In [28]:
# Use 255 outside the park and preserve 0/1 inside it.
osm_water_export = (
    osm_water_mask
    .unmask(255)
    .toByte()
)

task = ee.batch.Export.image.toDrive(
    image=osm_water_export,
    description="grand_teton_osm_surface_water_mask",
    folder="earth_engine_exports",
    fileNamePrefix="grand_teton_osm_surface_water_mask",
    region=park,
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True,
        "noData": 255,
    },
)

task.start()
print(task.status())

{'state': 'READY', 'description': 'grand_teton_osm_surface_water_mask', 'priority': 100, 'creation_timestamp_ms': 1785083730437, 'update_timestamp_ms': 1785083730437, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'F63JTAJV6RGGAA3TZ6XC6VIW', 'name': 'projects/teton-classifier/operations/F63JTAJV6RGGAA3TZ6XC6VIW'}


In [29]:
task.status()

{'state': 'RUNNING',
 'description': 'grand_teton_osm_surface_water_mask',
 'priority': 100,
 'creation_timestamp_ms': 1785083730437,
 'update_timestamp_ms': 1785083735697,
 'start_timestamp_ms': 1785083735638,
 'task_type': 'EXPORT_IMAGE',
 'attempt': 1,
 'id': 'F63JTAJV6RGGAA3TZ6XC6VIW',
 'name': 'projects/teton-classifier/operations/F63JTAJV6RGGAA3TZ6XC6VIW'}